# Elevation and mass change trends

This notebook calculates most of the aggregated trend values that are
cited in the associated article. Skip to the second half of the notebook
for the final results. About the first half is needed to prepare the
results.

In [19]:
from cryoswath.misc import (
    rgi_code_translator,
    rgi_o1region_translator,
    load_glacier_outlines,
)
import numpy as np
import pandas as pd
import pickle
from scipy.stats import t as t_dist
import statsmodels.api as sm
from textwrap import dedent, indent
from cryoswath_paper_workflow.workflow_support import (
    fit_deseasoned_trend,
    snakemake_fallback_builder,
    add_result,
    to_yaml,
    strip_spec_chars,
    o1o2reg_tuple_list,
)

snakemake_fallback = snakemake_fallback_builder(globals())

results = dict()

In [20]:
o2_areas = pd.read_csv(
    snakemake_fallback("input", "area_tbl", "data/L4/o2region__area_km2.csv"),
    index_col=["o1", "o2"],
).area_km2
o2_areas.index.set_names([None, None], inplace=True)

In [21]:
with open(
    snakemake_fallback(
        "input", "area_rgi", "resources/subregion_area_totals_from_RGI.pkl"
    ),
    "rb",
) as f:
    o2_areas_rgi = pickle.load(f)
o2_areas_rgi = o2_areas_rgi.loc[[o1 for o1, _ in o1o2reg_tuple_list]]

In [22]:
o2_area_diff_abs = (o2_areas_rgi - o2_areas.reindex_like(o2_areas_rgi).fillna(0)).abs()
tmp = o2_area_diff_abs.groupby(level=0).sum() / o2_areas.groupby(level=0).sum()
results.update(
    {
        "area_diff": {
            "min": {
                "name": rgi_code_translator(f"{tmp.idxmin():02d}"),
                "value": 100 * float(tmp.min()),
                "units": [r"\%"],
            },
            "max": {
                "name": rgi_code_translator(f"{tmp.idxmax():02d}"),
                "value": 100 * float(tmp.max()),
                "units": [r"\%"],
            },
        }
    }
)

In [23]:
def rgi_uncertainty(rgi_id: int | tuple[int, int]) -> np.floating:
    return (
        3
        * 0.039
        * load_glacier_outlines(
            f"0{rgi_id}"
            if isinstance(rgi_id, int)
            else f"{rgi_id[0]:02d}-{rgi_id[1]:02d}",
            "glaciers",
            union=False,
        ).area_km2  # pyright: ignore[reportAttributeAccessIssue]
        ** 0.70
    ).sum()

In [24]:
results["rgi_unc"] = {
    rgi_o1region_translator(o1): {
        "value": 100
        * float(rgi_uncertainty(o1) / o2_areas_rgi.groupby(level=0).sum().loc[o1]),
        "units": [r"\%"],
    }
    for o1, _ in o1o2reg_tuple_list
}

In [25]:
o2_elev_df = pd.read_csv(
    snakemake_fallback(
        "input",
        "avg_elev_tbl",
        "data/L4/Arctic_monthly_avg_glacier_surface_elevation_wrt_2010_per_RGI_o2region__meter.csv",
    ),
    index_col=0,
    header=[0, 1, 2],
    skiprows=[3],
    parse_dates=True,
).rename_axis("time")
o2_elev_df.columns = pd.MultiIndex.from_tuples(
    [tuple(int(ix) if len(ix) <= 2 else ix for ix in tup) for tup in o2_elev_df.columns]
)
o2_elev_df = o2_elev_df.swaplevel(1, 2, axis=1)
o2_elev_df = o2_elev_df.swaplevel(0, 1, axis=1)
o2_elev_df.dropna(axis=0, how="any", inplace=True)
o2_elev_df = o2_elev_df.iloc[2:]

In [26]:
o2_elev_df

,elevation,uncertainty,elevation,uncertainty,elevation,uncertainty,elevation,uncertainty,elevation,uncertainty,...,elevation,uncertainty,elevation,uncertainty,elevation,uncertainty,elevation,uncertainty,elevation,uncertainty
,1,1,1,1,1,1,1,1,1,1,...,8,8,8,8,9,9,9,9,9,9
,2,2,3,3,4,4,5,5,6,6,...,2,2,3,3,1,1,2,2,3,3
time,,,,,,,,,,,,,,,,,,,,,
2010-09-01,-1.475565,7.177011,-0.503287,0.764862,-1.894120,6.628814,-1.432199,6.416238,-2.082378,9.487535,...,-1.131115,9.748041,-2.200918,10.022948,-1.039683,3.975545,-0.387632,3.562936,-0.090955,3.105630
2010-10-01,-0.780921,7.287648,-0.475141,0.609860,-2.819500,6.520703,-1.892803,6.053938,-1.584374,9.334015,...,-1.809375,8.943087,-2.024839,9.526618,-0.918040,3.117213,-0.439558,2.287497,-0.287775,2.268904
2010-11-01,-0.828298,7.270514,-0.399743,14.441720,-1.780751,6.385156,-1.356671,5.972132,-1.630608,8.481242,...,-1.212448,8.696870,1.896317,9.540525,-0.558408,2.522065,-0.462685,2.124267,-0.451494,2.132149
2010-12-01,0.333060,7.217863,-0.049906,14.434530,-1.855929,6.494971,-0.583143,5.751751,-1.983996,8.470474,...,-0.150267,8.497508,0.776563,9.266446,-0.456632,2.516682,-0.211783,1.923601,-0.424244,1.889018
2011-01-01,0.355433,7.678929,-1.123677,10.388981,-1.057725,6.502943,-0.159072,5.853982,-1.791354,8.463123,...,-0.092221,8.461516,0.427696,9.303541,-0.380644,2.498122,-0.170539,1.912728,-0.455960,1.883097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-07-01,-12.441047,9.597682,-14.418684,0.446637,-17.625095,6.690452,-14.143600,5.811379,-16.779852,9.340713,...,-4.267473,7.170827,-8.573491,9.503825,-7.797391,2.784714,-8.843442,1.815267,-5.987721,2.141772


In [27]:
quarterly_per_o2 = (
    (o2_elev_df["elevation"].dropna(axis=1, how="all").dropna(axis=0, how="any"))
    .resample("QS-AUG", label="left")
    .first()
)

In [28]:
quarterly_per_o1 = (
    (
        (o2_elev_df["elevation"] * o2_areas)
        .dropna(axis=1, how="all")  # pyright: ignore[reportArgumentType]
        .dropna(axis=0, how="any")
        .transpose()
        .groupby(level=0)
        .sum()
        .transpose()
        / ((~o2_elev_df["elevation"].iloc[0, :].isnull()) * o2_areas)  # type: ignore
        .dropna()
        .groupby(level=0)
        .sum()
    )
    .resample("QS-AUG", label="left")
    .first()
)

In [29]:
elev_change_o1 = quarterly_per_o1[0::4].diff().shift(-1)[:-1]
elev_change_o1

,1,3,4,5,6,7,8,9
time,,,,,,,,
2010-08-01,-0.764849,-1.597735,-0.686939,-0.559219,0.157574,-1.224336,0.193417,-0.490276
2011-08-01,0.139663,-0.607348,-0.952140,-0.552097,-0.928857,0.207226,-0.065019,-0.387096
2012-08-01,-1.031961,0.381329,-0.315434,0.050921,-0.126111,-1.340567,-0.138060,-0.510859
2013-08-01,-1.216320,-0.628173,-0.802029,-0.449413,-0.576408,-0.854850,-0.128148,0.114994
2014-08-01,-1.801758,-0.596525,-0.620917,-0.192874,0.858739,-0.031635,-1.252135,-0.502513
2015-08-01,-1.005632,-0.015198,-0.834472,-0.816535,-1.055842,-0.647316,-0.322651,-0.799765
2016-08-01,-0.912831,0.317881,-0.509253,0.289179,-0.002884,-1.236648,0.157133,-0.217037
2017-08-01,-0.773643,0.137496,0.050243,-0.217341,-0.788251,-0.491670,-0.975294,-0.482305
2018-08-01,-2.177980,-0.864197,-1.607530,-0.747170,-0.901394,-1.004185,0.710570,-0.481860


In [30]:
with open(
    snakemake_fallback("input", "yr1_corr", "results/one_year_error_correlation.txt"),
    "r",
) as f:
    yr1_corr = float(f.read().strip())

In [31]:
annual_per_o1_err_sq = (
    ((o2_elev_df["uncertainty"] * o2_areas) ** 2)
    .dropna(axis=1, how="all")
    .transpose()
    .groupby(level=0)
    .sum()
    .transpose()
    ** 0.5
    / ((~o2_elev_df["uncertainty"].iloc[0, :].isnull()) * o2_areas)
    .dropna()
    .groupby(level=0)
    .sum()
).resample("QS-AUG", label="left").first()[0::4] ** 2
# print(annual_per_o1_err_sq)
elev_change_err_o1 = (
    annual_per_o1_err_sq
    + annual_per_o1_err_sq.shift(1)
    - 2 * yr1_corr * annual_per_o1_err_sq**0.5 * annual_per_o1_err_sq.shift(1) ** 0.5
).shift(-1).dropna() ** 0.5
elev_change_err_o1

,1,3,4,5,6,7,8,9
time,,,,,,,,
2010-08-01,2.955110,1.370684,1.821917,2.135524,5.175677,2.843482,4.426956,1.508649
2011-08-01,2.812534,1.232095,1.656145,1.887627,2.271763,2.129028,3.685951,0.935190
2012-08-01,2.855550,1.245240,1.660520,1.889721,2.179727,2.125961,3.748820,0.936523
2013-08-01,2.957308,1.260659,1.644268,1.890814,2.230369,2.152804,3.552567,0.939253
2014-08-01,2.942043,1.255466,1.674789,1.898206,2.402356,2.175260,3.692062,0.982164
2015-08-01,2.934000,1.245943,1.674689,1.906331,2.596612,2.142313,3.875512,0.993756
2016-08-01,2.987568,1.249223,1.661644,1.929128,2.597993,2.098679,3.983529,0.983730
2017-08-01,2.937513,1.266948,1.690255,1.948266,2.609635,2.086528,3.819044,0.960129
2018-08-01,2.905181,1.262553,1.666492,1.947636,2.589756,2.183581,3.736572,0.933293


In [32]:
area_per_o1 = (
    o2_areas.reindex_like(o2_areas_rgi.loc[o2_areas.index.levels[0]])
    .fillna(o2_areas_rgi)
    .groupby(level=0)
    .sum()
)

In [33]:
mass_change_o1 = elev_change_o1 * area_per_o1 * 1e6 * 850 / 1000 / 1e9
mass_change_o1

,1,3,4,5,6,7,8,9
time,,,,,,,,
2010-08-01,-55.719991,-136.794110,-23.626019,-42.006506,1.452051,-33.520461,0.482856,-20.492410
2011-08-01,10.174614,-51.999641,-32.747112,-41.471538,-8.559475,5.673544,-0.162317,-16.179736
2012-08-01,-75.179421,32.648484,-10.848761,3.824990,-1.162124,-36.702683,-0.344659,-21.352712
2013-08-01,-88.610142,-53.782583,-27.584311,-33.758292,-5.311639,-23.404480,-0.319915,4.806475
2014-08-01,-131.259872,-51.072992,-21.355307,-14.487982,7.913338,-0.866111,-3.125892,-21.003899
2015-08-01,-73.261323,-1.301242,-28.700132,-61.335191,-9.729652,-17.722520,-0.805482,-33.428330
2016-08-01,-66.500679,27.216189,-17.514818,21.722093,-0.026573,-33.857539,0.392276,-9.071654
2017-08-01,-56.360654,11.772055,1.728024,-16.325855,-7.263782,-13.461185,-2.434772,-20.159219
2018-08-01,-158.668050,-73.990432,-55.288051,-56.124726,-8.306407,-27.493060,1.773903,-20.140652


In [ ]:
mass_change_o1_err = (
    (
        elev_change_err_o1**2
        + (150 / 850 * elev_change_o1) ** 2
        + (o2_area_diff_abs**2).groupby(level=0).sum()
        * (elev_change_o1 / area_per_o1) ** 2
        + pd.Series({i: rgi_uncertainty(i) for i in area_per_o1.index}) ** 2
        * (elev_change_o1 / area_per_o1) ** 2
    )
    ** 0.5
    * area_per_o1
    * 1e6
    * 850
    / 1000
    / 1e9
)
mass_change_o1_err

,1,3,4,5,6,7,8,9
time,,,,,,,,
2010-08-01,215.537700,119.849335,62.819547,160.632599,47.695806,78.093782,11.054436,63.183599
2011-08-01,204.935934,105.929852,57.273862,142.035333,20.990976,58.323824,9.204758,39.228294
2012-08-01,208.483907,106.812172,57.164040,142.005472,20.089575,58.590399,9.361812,39.360662
2013-08-01,216.040085,108.392631,56.782279,142.210800,20.576506,59.110362,8.872016,39.303044
2014-08-01,215.609364,107.908898,57.745668,142.663931,22.183861,59.580641,9.236451,41.252774
2015-08-01,214.166214,106.717106,57.841289,143.659428,23.991311,58.762080,9.678843,41.986527
2016-08-01,217.993670,107.105247,57.254104,145.013587,23.942521,57.794359,9.947625,41.182454
2017-08-01,214.262546,108.534382,58.155081,146.428287,24.083932,57.201497,9.546547,40.322926
2018-08-01,213.520379,108.923851,58.161474,146.687414,23.911594,60.004634,9.336296,39.206463


In [ ]:
# importance of elevation change uncertainty
tmp = elev_change_err_o1 / elev_change_o1.abs()
print(tmp)
# specifically in the "high-mass-loss" year 2018/19
tmp.loc["2018"].T.describe()

                    1          3          4          5           6          7  \
time                                                                            
2010-08-01   3.863652   0.857892   2.652224   3.818764   32.846082   2.322468   
2011-08-01  20.137954   2.028646   1.739392   3.419014    2.445762  10.273920   
2012-08-01   2.767109   3.265523   5.264244  37.110952   17.284142   1.585867   
2013-08-01   2.431357   2.006868   2.050136   4.207297    3.869425   2.518342   
2014-08-01   1.632873   2.104633   2.697283   9.841704    2.797538  68.761759   
2015-08-01   2.917567  81.979082   2.006885   2.334659    2.459280   3.309533   
2016-08-01   3.272858   3.929843   3.262905   6.671052  900.944136   1.697070   
2017-08-01   3.796989   9.214442  33.641440   8.964110    3.310666   4.243753   
2018-08-01   1.333888   1.460955   1.036678   2.606685    2.873055   2.174481   
2019-08-01   4.004006   3.548333   1.666385  12.805395    5.369021   2.562194   
2020-08-01   3.424777  23.02

time,2018-08-01
count,8.000000
mean,2.335144
std,1.338794
min,1.036678
25%,1.429188
50%,2.055667
75%,2.673277
max,5.258554


In [ ]:
def precision(val):
    return max(0, round(1 - np.log10(val)))

In [ ]:
mass_change_arctic = mass_change_o1.sum(axis=1)
mass_change_arctic

In [ ]:
mass_change_arctic_err = (mass_change_o1_err**2).sum(axis=1) ** 0.5
mass_change_arctic_err

In [ ]:
min_year_idx = mass_change_arctic.idxmin()
year = min_year_idx.year
tmp_val = mass_change_arctic.loc[min_year_idx]
tmp_err = mass_change_arctic_err.loc[min_year_idx]
add_result(results, "arctic_mass_18_19", tmp_val, tmp_err, units=["Gt"])
print(f"Arctic-wide mass balance minimum in {year}/{year + 1}:")
print(
    f"${tmp_val:.{precision(tmp_err)}f}\\pm{tmp_err:.{precision(tmp_err)}f}$~\\unit{{Gt}}~\\unit{{yr^{{-1}}}}"
)

In [ ]:
min_region_idx = mass_change_o1.loc[min_year_idx].idxmin()
tmp_val = mass_change_o1.loc[min_year_idx, min_region_idx]
tmp_err = mass_change_o1_err.loc[min_year_idx, min_region_idx]
add_result(results, "alaska_mass_18_19", tmp_val, tmp_err, units=["Gt"])
print(
    rgi_code_translator(f"{min_region_idx:02d}"),
    f"mass balance minimum in {year}/{year + 1}:",
)
print(
    f"${tmp_val:.{precision(tmp_err)}f}\\pm{tmp_err:.{precision(tmp_err)}f}$~\\unit{{Gt}}~\\unit{{yr^{{-1}}}}"
)

In [ ]:
tmp_regions = [3, 4, 5]
for o1, short_name in zip(tmp_regions, ["CNorth", "CSouth", "grnl"]):
    tmp_val = mass_change_o1.loc["2018", o1].item()
    tmp_err = mass_change_o1_err.loc["2018", o1].item()
    add_result(results, short_name + "_mass_18_19", tmp_val, tmp_err, units=["Gt"])
    print(rgi_code_translator(f"{o1:02d}"), f"mass balance in {year}/{year + 1}:")
    print(
        f"${tmp_val:.{precision(tmp_err)}f}\\pm{tmp_err:.{precision(tmp_err)}f}$~\\unit{{Gt}}~\\unit{{yr^{{-1}}}}"
    )
tmp_val = mass_change_o1.loc["2018", tmp_regions].sum(axis=1).item()
tmp_err = (mass_change_o1_err.loc["2018", tmp_regions] ** 2).sum(axis=1).item() ** 0.5
add_result(results, "cncsgrnl_mass_18_19", tmp_val, tmp_err, units=["Gt"])
print(
    "Collective mass change in 2018/19 across",
    ", ".join([rgi_code_translator(f"{o1:02d}") for o1 in tmp_regions]),
)
print(
    f"${tmp_val:.{precision(tmp_err)}f}\\pm{tmp_err:.{precision(tmp_err)}f}$~\\unit{{Gt}}~\\unit{{yr^{{-1}}}}"
)

In [ ]:
elev_change_err_arctic = (
    (elev_change_err_o1 * o2_areas.groupby(level=0).sum()) ** 2
).sum(axis=1) ** 0.5 / ((~o2_elev_df["elevation"].iloc[0, :].isnull()) * o2_areas).sum()
elev_change_err_arctic

In [ ]:
def trend_and_unc_from_fit(fit_res):
    return np.array([fit_res.params.trend, 2 * fit_res.bse.trend]) * (
        4 if fit_res.resid.index.freq.freqstr[0] == "Q" else 1
    )

In [ ]:
def trend_difference_pvalue(first_fit, second_fit):
    # Two-sided Welch–Satterthwaite test, neglecting covariance between fits.
    variances = np.array([first_fit.bse.trend**2, second_fit.bse.trend**2])
    variance = variances.sum()
    dof = (
        variance**2
        / (variances**2 / np.array([first_fit.df_resid, second_fit.df_resid])).sum()
    )
    statistic = abs(first_fit.params.trend - second_fit.params.trend) / variance**0.5
    return float(2 * t_dist.sf(statistic, df=dof))

In [ ]:
def mass_and_unc(trend, trend_unc, rgi_id: int | tuple[int, int]):
    if isinstance(rgi_id, int):
        area = area_per_o1.loc[rgi_id]
        area_diff = o2_area_diff_abs.groupby(level=0).sum().loc[rgi_id]
        if rgi_id == 1:
            area_diff += pd.read_csv(
                snakemake_fallback(
                    "input", "omitted_tbl", "results/omitted_subregions.csv"
                ),
                index_col=0,
            ).at["01-03", "area_km2"]
    else:
        area = o2_areas.loc[rgi_id]
        area_diff = o2_area_diff_abs.loc[rgi_id]
    mass_trend = trend * area * 1e6 * 850 / 1000 / 1e9
    rel_unc = np.array(
        [
            trend_unc / trend,
            60 / 850,
            rgi_uncertainty(rgi_id) / area,
            area_diff / area,
        ]
    )
    return np.array([np.sign(mass_trend), (rel_unc**2).sum() ** 0.5]) * np.abs(
        mass_trend
    )

In [ ]:
def compute_average_trend(o1region, timeslice, _plot=False, _print=False):
    fit_res = fit_deseasoned_trend(quarterly_per_o1.loc[timeslice, o1region])
    trend, trend_unc = trend_and_unc_from_fit(fit_res)
    mass_trend, mass_trend_unc = mass_and_unc(trend, trend_unc, o1region)
    if _print:
        if not isinstance(o1region, slice):
            print(f"RGI o1: {o1region:02d}")
        else:
            print("Arctic-wide")
        print(f"Surface trend: {trend:.2f} +- {trend_unc:.2f} m/yr")
        print(f"Mass trend: {mass_trend:.1f} +- {mass_trend_unc:.1f} Gt/yr")
        print("(uncertainty from fit stats and density)")
        print("Student t =", fit_res.tvalues.trend)
        print("p =", fit_res.pvalues.trend)
    if _plot:
        import matplotlib.pyplot as plt

        sm.graphics.plot_fit(fit_res, 1)
        plt.show()
    if _print:
        print(fit_res.summary())
    return (
        trend,
        trend_unc,
        mass_trend,
        mass_trend_unc,
        fit_res.pvalues.trend,
    )

In [ ]:
timeslices = [
    slice("2010-08", "2014-09"),
    slice("2020-08", "2024-09"),
    slice(None),
    slice("2010", "2019"),
    slice("2010-08", "2020-08"),
]

In [ ]:
def timeslice_to_index(_slice):
    return "_".join(
        [("_" if x is None else x[2:4]) for x in [_slice.start, _slice.stop]]
    )

In [ ]:
col_names = ["specific", "sp_err", "mass", "m_err", "p-val"]

In [ ]:
o1_trend_df = pd.DataFrame(
    index=rgi_o1region_translator(quarterly_per_o1.columns.tolist()),
    columns=pd.MultiIndex.from_product(
        [[timeslice_to_index(t) for t in timeslices], col_names]
    ),
)
for timeslice in timeslices:
    o1_trend_df.loc(1)[timeslice_to_index(timeslice)] = [
        compute_average_trend(o1region, timeslice, _plot=True, _print=True)
        for o1region in quarterly_per_o1.columns
    ]
o1_trend_df.loc["Arctic-wide", (slice(None), "specific")] = o1_trend_df.loc(1)[
    slice(None), "specific"
].apply(np.average, weights=area_per_o1)
o1_trend_df.loc["Arctic-wide", (slice(None), "sp_err")] = (
    (
        o1_trend_df.iloc(0)[:-1].loc(1)[slice(None), "sp_err"]
        * area_per_o1.values[:, np.newaxis]
    )
    ** 2
).sum() ** 0.5 / area_per_o1.sum()
o1_trend_df.loc["Arctic-wide", (slice(None), "mass")] = o1_trend_df.loc(1)[
    slice(None), "mass"
].sum(axis=0)
o1_trend_df.loc["Arctic-wide", (slice(None), "m_err")] = (
    (o1_trend_df.loc(1)[slice(None), "m_err"] ** 2).sum(axis=0)
) ** 0.5
o1_trend_df[pd.MultiIndex.from_product([["ref"], ["mass", "m_err"]])] = [
    [-78.9, 6.5],
    [-32.0, 2.7],
    [-29.4, 2.4],
    [-35.1, 4.3],
    [-7.7, 0.6],
    [-13.3, 1.1],
    [-1.5, 0.2],
    [-11.8, 1.1],
    [-210, 9],
]

print(o1_trend_df)

In [ ]:
o1_trend_df.to_csv(
    snakemake_fallback("output", "trend_tbl", "results/result__regional_trends.csv")
)

Computing p-values for differences between fitted trends in 2010–2014 and 2020–2024.
The two-sided Welch–Satterthwaite comparison uses slope standard errors and residual
degrees of freedom from the existing OLS fits, assuming independent fit errors.
Within each region, mass trends use the same area and density factors, so equality
of mass trends is equivalent to equality of elevation trends. Shared conversion
uncertainties are therefore excluded from this test. Arctic-wide trends are fitted
to the area-weighted regional series. Significance level: 0.05.

In [ ]:
comparison_series = quarterly_per_o1.copy()
comparison_series.columns = rgi_o1region_translator(comparison_series.columns.tolist())
comparison_series["Arctic-wide"] = (
    quarterly_per_o1.mul(area_per_o1, axis=1).sum(axis=1) / area_per_o1.sum()
)
comparison_series

In [ ]:
p_vals = pd.Series(index=comparison_series.columns, dtype=float, name="p-values")
for region, series in comparison_series.items():
    fits = [fit_deseasoned_trend(series.loc[timeslice]) for timeslice in timeslices[:2]]
    p_vals[region] = trend_difference_pvalue(*fits)
p_vals

In [ ]:
p_vals.rename(index={"Svalbard and Jan Mayen": "Svalbard"}).rename(
    "p_value"
).rename_axis("region").to_csv(
    snakemake_fallback(
        "output",
        "period_p_values",
        "results/result__period_comparison_p_values.csv",
    )
)

In [ ]:
results

In [ ]:
def compute_average_trend_o2(o2region, timeslice, _plot=False, _print=False):
    fit_res = fit_deseasoned_trend(quarterly_per_o2.loc[timeslice, o2region])
    trend, trend_unc = trend_and_unc_from_fit(fit_res)
    mass_trend, mass_trend_unc = mass_and_unc(trend, trend_unc, o2region)
    if _print:
        if not isinstance(o2region, slice):
            print("RGI o2: " + " ,".join([f"{x:02d}" for x in o2region]))
        print(f"Surface trend: {trend:.2f} +- {trend_unc:.2f} m/yr")
        print(f"Mass trend: {mass_trend:.1f} +- {mass_trend_unc:.1f} Gt/yr")
        print("(uncertainty from fit stats and density)")
        print("Student t =", fit_res.tvalues.trend)
        print("p =", fit_res.pvalues.trend)
    if _plot:
        import matplotlib.pyplot as plt

        sm.graphics.plot_fit(fit_res, 1)
        plt.show()
    if _print:
        print(fit_res.summary())
    return (
        trend,
        trend_unc,
        mass_trend,
        mass_trend_unc,
        fit_res.pvalues.trend,
    )

In [ ]:
grnl_trend_df = pd.DataFrame(
    index=rgi_code_translator(
        [f"05-{o2:02d}" for o2 in quarterly_per_o2[5].columns.tolist()]
    ),
    columns=pd.MultiIndex.from_product(
        [[timeslice_to_index(t) for t in timeslices], col_names]
    ),
)
for timeslice in timeslices:
    grnl_trend_df.loc(1)[timeslice_to_index(timeslice)] = [
        compute_average_trend_o2((5, o2), timeslice)
        for o2 in quarterly_per_o2[5].columns
    ]

# print(grnl_trend_df)

In [ ]:
grnl_trend_df.to_csv(
    snakemake_fallback("output", "grnl_tbl", "results/result__greenland_trends.csv")
)

Helper functions to format the results as latex table:

In [ ]:
def format_latex_cell(val, err):
    return f"{val:.{precision(err)}f} & {err:.{precision(err)}f}"

In [ ]:
def wrap_in_multicol(tex):
    width = 2 if "&" in tex else 1
    col_fmt = r">{$}r<{$}@{ $\pm$ }>{$}l<{$}" if "&" in tex else "l"
    out = f"""\
        \\multicolumn{{{width}}}{{r}}{{
            \\scriptsize %\\itshape
            \\begin{{tabular}}{{{col_fmt}}}\n{
        indent(
            tex,
            "\
                ",
        )
    }       
            \\end{{tabular}}
        }}"""
    return dedent(out)

In [ ]:
def format_latex_grnl_cols(df):
    return " \\\\ \n".join(
        df.str.replace("South East", "Southeast").str.replace("South West", "Southwest")
        if isinstance(df, pd.Index)
        else [format_latex_cell(*_row[1]) for _row in df.iterrows()]
    )

In [ ]:
def format_latex_grnl_inset(df):
    return "% Greenland subregions \n" + "\n&\n".join(
        map(
            wrap_in_multicol,
            map(
                format_latex_grnl_cols,
                [
                    df.index,
                    df.loc(1)["___", ["mass", "m_err"]],
                    df.loc(1)["___", ["specific", "sp_err"]],
                    df.loc(1)["10_14", ["mass", "m_err"]],
                    df.loc(1)["20_24", ["mass", "m_err"]],
                    df.loc(1)["10_19", ["mass", "m_err"]],
                ],
            ),
        )
    )


# print(format_latex_grnl_inset(grnl_trend_df))

In [ ]:
def format_latex_table_row(row):
    return " & ".join(
        [
            row.name,
            *[
                format_latex_cell(*_row)
                for _row in [
                    row.loc["___", ["mass", "m_err"]],
                    row.loc["___", ["specific", "sp_err"]],
                    row.loc["10_14", ["mass", "m_err"]],
                    row.loc["20_24", ["mass", "m_err"]],
                    row.loc["10_19", ["mass", "m_err"]],
                    row.loc["ref", ["mass", "m_err"]],
                ]
            ],
        ]
    )

In [ ]:
def format_latex_table(df):
    return [format_latex_table_row(row) for _, row in df.iterrows()]

In [ ]:
header = r"""
\begin{tabular}{@{}l*{5}{@{\;\;}>{$}r<{$}@{ $\pm$ }>{$}l<{$}}|@{\;\;}>{$}r<{$}@{ $\pm$ }>{$}l<{$}@{}}
% first header row
    % empty
    & \multicolumn{2}{c}{Mass balance} & \multicolumn{2}{c}{Surface trend}
    & \multicolumn{4}{c}{Period comparison}
    & \multicolumn{4}{c}{Reference comparison}
    \\
% second header row
    % empty
    & \multicolumn{4}{c}{Aug 2010 -- Nov 2024}
    & \multicolumn{2}{c}{2010 -- 2014} & \multicolumn{2}{c}{2020 -- 2024}
    & \multicolumn{4}{c}{2010 -- 2019}
    \\
% third header row
    Region %\quad \textbackslash \quad Period 
    & \multicolumn{2}{c}{\unit{Gt}~\unit{yr^{-1}}}
    & \multicolumn{2}{c}{\unit{m}~\unit{yr^{-1}}}
    & \multicolumn{2}{c}{\unit{Gt}~\unit{yr^{-1}}}
    & \multicolumn{2}{c}{\unit{Gt}~\unit{yr^{-1}}}
    & \multicolumn{2}{c}{\unit{Gt}~\unit{yr^{-1}}}
    & \multicolumn{2}{c}{\unit{Gt}~\unit{yr^{-1}}}
    \\ \hline
""".strip()
# print(header)

In [ ]:
with open(
    snakemake_fallback(
        "output", "subregion_trend_table", "results/tab01__subregion_trend_table.tex"
    ),
    "w",
) as f:
    print(header, file=f)
    print(
        *format_latex_table(o1_trend_df.iloc[:4]),
        format_latex_grnl_inset(grnl_trend_df),
        *format_latex_table(o1_trend_df.iloc[4:]),
        sep=" \\\\\n",
        file=f,
    )
    print(r"\end{tabular}", end="", file=f)

In [ ]:
# quick excursion for Jakob and Gourmelen 2023 comarison:
# --before vs after PDO transition--
# [
#     compute_average_trend(1, timeslice)
#     for timeslice in [slice(), slice()]
# ]
results.update(
    {
        f"alaska_mass_{start[2:4]}_{stop[2:4]}": {
            **{
                k: float(v)
                for k, v in zip(
                    ["_", "_", "value", "uncertainty", "_"],
                    compute_average_trend(1, slice(start, stop)),
                )
                if k != "_"
            },
            "units": ["Gt", "yr^{-1}"],
        }
        for start, stop in [("2010", "2019-08"), ("2019-08", "2024-11")]
    }
)
# Keep both period endpoints, including the shared August 2019 observation.
# As in the regional comparisons, covariance between fits is neglected.
results.update(
    {
        "alaska_p_1019_1924": {
            "value": trend_difference_pvalue(
                *[
                    fit_deseasoned_trend(quarterly_per_o1.loc[start:stop, 1])
                    for start, stop in [("2010", "2019-08"), ("2019-08", "2024-11")]
                ]
            )
        }
    }
)

In [ ]:
to_yaml(
    results,
    snakemake_fallback(
        "output", "value_collection", "results/result__value_collection.yaml"
    ),
)